# Add NeMo Relay to a NemoClaw sandbox

This optional notebook installs [NeMo Relay](https://github.com/NVIDIA/NeMo-Relay) into an existing NemoClaw OpenClaw sandbox and writes agent trajectories in ATIF format. Complete `deploy_nemoclaw.ipynb` first, run this notebook before using the agent, then continue with `deploy_vss_orchestrator.ipynb`. Run this notebook again after recreating the sandbox.

Do not run this notebook for Hermes. Hermes already includes NeMo Relay in its runtime.

**Data handling:** prompt text, per-call response text, tool arguments, and tool results are excluded by the configuration below. NeMo Relay 0.7.3 still writes the agent's final message in the session-close summary. Do not enable it when that output must not be stored on disk. A trajectory is written when the agent session closes or the gateway restarts.

The pinned packages and licenses are listed in `nemoclaw/LICENSE-3rd-party.txt`.


## 1. Settings

Set the sandbox name used in `deploy_nemoclaw.ipynb`. Leave `RELAY_OBSERVABILITY` true to install or refresh Relay. Set it to false and rerun the notebook to disable an earlier installation.


In [ ]:
import json
import os
import shlex
import shutil
import subprocess
import time


# ================== Operator settings ==================
NEMOCLAW_SANDBOX_NAME = "demo"
NEMOCLAW_DASHBOARD_PORT = 18789
RELAY_OBSERVABILITY = True
RELAY_ATIF_DIR = "/sandbox/.openclaw/nemo-relay-atif"

# Keep this pin aligned with nemoclaw/LICENSE-3rd-party.txt.
RELAY_RELEASE = "0.7.3"


# ================== Derived (no need to touch) ==================
SHELL_ENV = globals().setdefault("_NOTEBOOK_SHELL_ENV", dict(os.environ))
NEMOCLAW_SANDBOX_NAME = SHELL_ENV.get("NEMOCLAW_SANDBOX_NAME", NEMOCLAW_SANDBOX_NAME).strip()
NEMOCLAW_DASHBOARD_PORT = int(SHELL_ENV.get("NEMOCLAW_DASHBOARD_PORT", str(NEMOCLAW_DASHBOARD_PORT)))
RELAY_OBSERVABILITY = str(SHELL_ENV.get("RELAY_OBSERVABILITY", RELAY_OBSERVABILITY)).strip().lower() == "true"
RELAY_ATIF_DIR = SHELL_ENV.get("RELAY_ATIF_DIR", RELAY_ATIF_DIR).strip()
RELAY_PLUGIN_PACKAGE = f"npm:nemo-relay-openclaw@{RELAY_RELEASE}"

if not NEMOCLAW_SANDBOX_NAME:
    raise ValueError("NEMOCLAW_SANDBOX_NAME cannot be empty")
if not 1024 <= NEMOCLAW_DASHBOARD_PORT <= 65535:
    raise ValueError("NEMOCLAW_DASHBOARD_PORT must be between 1024 and 65535")
if not RELAY_ATIF_DIR.startswith("/"):
    raise ValueError("RELAY_ATIF_DIR must be an absolute sandbox path")

print(
    f"Sandbox: {NEMOCLAW_SANDBOX_NAME}\n"
    f"Dashboard port: {NEMOCLAW_DASHBOARD_PORT}\n"
    f"Relay enabled: {RELAY_OBSERVABILITY}\n"
    f"Relay package: {RELAY_PLUGIN_PACKAGE}\n"
    f"ATIF directory: {RELAY_ATIF_DIR}"
)


## 2. Verify the existing sandbox

This cell checks the NemoClaw command, the named sandbox, and the OpenClaw and npm commands inside it. It does not create or replace a sandbox.


In [ ]:
def run(command: list[str], *, timeout: int = 120, check: bool = True) -> subprocess.CompletedProcess[str]:
    print("$", shlex.join(command), flush=True)
    result = subprocess.run(command, capture_output=True, text=True, timeout=timeout)
    for stream in (result.stdout, result.stderr):
        if stream:
            print(stream, end="" if stream.endswith("\n") else "\n", flush=True)
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, command)
    return result


def sandbox_exec(script: str, *, timeout: int = 120, check: bool = True) -> subprocess.CompletedProcess[str]:
    return run(
        [
            "nemoclaw",
            NEMOCLAW_SANDBOX_NAME,
            "exec",
            "--timeout",
            str(timeout),
            "--",
            "sh",
            "-lc",
            script,
        ],
        timeout=timeout + 30,
        check=check,
    )


def gateway_healthy(timeout_s: int = 90, poll_s: int = 3, attempt_s: int = 15) -> bool:
    health_url = f"http://127.0.0.1:{NEMOCLAW_DASHBOARD_PORT}/health"
    probe = (
        f'code=$(curl -s -o /dev/null -w "%{{http_code}}" --max-time 5 {health_url}); '
        'case "$code" in 2*|401|403) exit 0;; *) exit 1;; esac'
    )
    deadline = time.monotonic() + timeout_s
    while True:
        try:
            result = sandbox_exec(probe, timeout=attempt_s, check=False)
        except subprocess.TimeoutExpired:
            result = None
        if result is not None and result.returncode == 0:
            return True
        if time.monotonic() >= deadline:
            return False
        time.sleep(poll_s)


def apply_gateway_config(reason: str) -> None:
    if run(["nemoclaw", NEMOCLAW_SANDBOX_NAME, "gateway", "restart"], check=False).returncode == 0:
        return
    print(f"Gateway restart reported a failure while applying {reason}; trying sandbox recovery.", flush=True)
    if run(["nemoclaw", NEMOCLAW_SANDBOX_NAME, "recover"], check=False).returncode == 0:
        return
    if not gateway_healthy():
        raise RuntimeError(f"gateway is down after applying {reason}")
    print("Gateway is answering its health probe; continuing.", flush=True)


if shutil.which("nemoclaw") is None:
    raise RuntimeError("Missing host command: nemoclaw. Run deploy_nemoclaw.ipynb first.")
run(["nemoclaw", NEMOCLAW_SANDBOX_NAME, "status", "--json"])
sandbox_exec('for cmd in openclaw npm node grep sed; do command -v "$cmd" >/dev/null || exit 1; done')
print("NEMO_RELAY_PREFLIGHT: passed")


## 3. Install and configure Relay

With `RELAY_OBSERVABILITY=True`, the cell applies NemoClaw's built-in npm egress preset, installs the pinned OpenClaw plugin and native binding, writes the Relay configuration, restarts the gateway, and verifies that the plugin loaded. Re-running it refreshes the current plugin generation.

With `RELAY_OBSERVABILITY=False`, it disables an earlier Relay installation and restarts the gateway. It leaves the npm preset in place because NemoClaw onboarding owns that preset.


In [ ]:
if not RELAY_OBSERVABILITY:
    installed = sandbox_exec(
        "openclaw plugins inspect nemo-relay --json >/dev/null 2>&1",
        check=False,
    )
    if installed.returncode == 0:
        sandbox_exec("openclaw plugins disable nemo-relay")
        apply_gateway_config("NeMo Relay disable")
    print("NEMO_RELAY_READY: disabled")
else:
    run(["nemoclaw", NEMOCLAW_SANDBOX_NAME, "policy", "add", "npm", "--yes"])

    install = f"openclaw plugins install {shlex.quote(RELAY_PLUGIN_PACKAGE)} --force"
    sandbox_exec(install, timeout=600)

    # The OpenClaw installer skips the optional platform binding. Resolve the
    # active generation from the plugin itself because --force leaves older
    # generations on disk. Installing into a searched path can select stale code.
    binding = (
        "set -eu\n"
        "pkg=\"$(openclaw plugins inspect nemo-relay --json"
        " | grep -o '\"installPath\": *\"[^\"]*\"' | head -1"
        " | sed 's/.*: *\"//; s/\"$//')\"\n"
        "[ -n \"$pkg\" ] || { echo 'nemo-relay install path not found' >&2; exit 2; }\n"
        "cd \"${pkg%/node_modules/nemo-relay-openclaw}\"\n"
        "npm install --no-save --no-audit --no-fund "
        f"\"nemo-relay-node-linux-$(node -p process.arch)-gnu@{RELAY_RELEASE}\"\n"
    )
    sandbox_exec(binding, timeout=600)

    relay_entry = {
        "enabled": True,
        "hooks": {"allowConversationAccess": True},
        "config": {
            "enabled": True,
            "backend": "hooks",
            "plugins": {
                "version": 1,
                "components": [
                    {
                        "kind": "observability",
                        "enabled": True,
                        "config": {
                            "version": 3,
                            "atif": {
                                "enabled": True,
                                "agent_name": "nemoclaw",
                                "output_directory": RELAY_ATIF_DIR,
                            },
                            "opentelemetry": {"enabled": False},
                        },
                    }
                ],
            },
            "capture": {
                "includePrompts": False,
                "includeResponses": False,
                "stripToolArgs": True,
                "stripToolResults": True,
            },
        },
    }
    run(
        [
            "nemoclaw",
            NEMOCLAW_SANDBOX_NAME,
            "config",
            "set",
            "--key",
            "plugins.entries.nemo-relay",
            "--value",
            json.dumps(relay_entry),
            "--config-accept-new-path",
        ]
    )
    apply_gateway_config("NeMo Relay configuration")
    sandbox_exec("openclaw plugins inspect nemo-relay --runtime --json >/dev/null")
    print(f"NEMO_RELAY_READY: enabled; ATIF directory={RELAY_ATIF_DIR}")


## 4. Read the trajectories

Use the agent normally. Close its session or restart the gateway before checking for a new trajectory. List the files from the host with:

```bash
nemoclaw <sandbox> exec -- sh -lc 'find /sandbox/.openclaw/nemo-relay-atif -maxdepth 1 -type f -print'
```

If you changed `RELAY_ATIF_DIR`, use that path instead. Keep the ATIF files out of source control because they can contain the agent's final response.
